# Entrega — Evaluación empírica de RitaAgent para Connect-4

Este notebook compara dos versiones del agente:

- **RitaVersion1**: MCTS + UCB con rollouts aleatorios.
- **RitaVersion2**: MCTS + UCB mejorado con más rollouts, default policy táctica, expansión ordenada, heurística de ventanas, detección de jugadas suicidas y amenazas dobles.

El estudio genera gráficas para:
1. V1 y V2 contra jugador aleatorio.
2. Desempeño por color.
3. Comparación directa V1 vs V2.
4. Sensibilidad al valor de `exploration_c`.
5. Sensibilidad al número de rollouts (`num_iterations`).
6. Ablation study de mejoras de V2.


## 1. Configuración de rutas

Este notebook ya viene configurado con tus rutas absolutas en macOS.

Si mueves el proyecto, cambia `PROJECT_ROOT`, `V1_PATH` y `V2_PATH`.


In [ ]:
from pathlib import Path
import sys
import time
import inspect
import importlib.util
import math
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(r"/Users/ritatrindadedacruz/Desktop/Universidad/Séptimo/IA/Proyecto Final/AI project/AI-Final-Project")
V1_PATH = Path(r"/Users/ritatrindadedacruz/Desktop/Universidad/Séptimo/IA/Proyecto Final/AI project/AI-Final-Project/tournament/groups/Group A/rita_v1.py")
V2_PATH = Path(r"/Users/ritatrindadedacruz/Desktop/Universidad/Séptimo/IA/Proyecto Final/AI project/AI-Final-Project/tournament/groups/Group A/rita_v2.py")

# Para que Python encuentre connect4/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Algunas versiones del proyecto tienen el código dentro de tournament/
TOURNAMENT_ROOT = PROJECT_ROOT / "tournament"
if TOURNAMENT_ROOT.exists() and str(TOURNAMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(TOURNAMENT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("V1_PATH:", V1_PATH, "exists:", V1_PATH.exists())
print("V2_PATH:", V2_PATH, "exists:", V2_PATH.exists())
print("sys.path[0:3]:", sys.path[:3])


## 2. Carga dinámica de Rita V1 y Rita V2

Esta sección carga tus archivos `rita_v1.py` y `rita_v2.py` sin depender de imports relativos.


In [ ]:
def load_module_from_path(path: Path, module_name: str):
    path = Path(path).resolve()

    if not path.exists():
        raise FileNotFoundError(f"No existe el archivo: {path}")

    spec = importlib.util.spec_from_file_location(module_name, str(path))

    if spec is None or spec.loader is None:
        raise ImportError(f"No se pudo cargar el módulo desde {path}")

    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)

    return module


def find_policy_class(module, preferred_names=None):
    preferred_names = preferred_names or []

    for name in preferred_names:
        if hasattr(module, name):
            cls = getattr(module, name)
            if inspect.isclass(cls) and hasattr(cls, "mount") and hasattr(cls, "act"):
                return cls

    candidates = []

    for _, obj in inspect.getmembers(module, inspect.isclass):
        if obj.__module__ != module.__name__:
            continue

        if hasattr(obj, "mount") and hasattr(obj, "act"):
            candidates.append(obj)

    if not candidates:
        raise ValueError("No se encontró una clase con mount() y act().")

    return sorted(candidates, key=lambda cls: cls.__name__)[0]


v1_module = load_module_from_path(V1_PATH, "rita_v1_module")
v2_module = load_module_from_path(V2_PATH, "rita_v2_module")

RitaV1Class = find_policy_class(v1_module, preferred_names=["RitaVersion1", "RitaAgentV1"])
RitaV2Class = find_policy_class(v2_module, preferred_names=["RitaVersion2", "RitaVersion1", "RitaAgentV2"])

print("Clase V1:", RitaV1Class.__name__)
print("Clase V2:", RitaV2Class.__name__)


## 3. Entorno local de Connect-4

Se implementa una simulación local para poder evaluar muchas partidas automáticamente.

Convención:
- `0`: casilla vacía
- `-1`: primer jugador
- `1`: segundo jugador


In [ ]:
ROWS = 6
COLS = 7
EMPTY = 0
P1 = -1
P2 = 1

def new_board():
    return np.zeros((ROWS, COLS), dtype=int)

def legal_actions(board):
    return [c for c in range(COLS) if board[0, c] == EMPTY]

def apply_move(board, col, player):
    if col not in legal_actions(board):
        raise ValueError(f"Acción ilegal: columna {col}. Legales: {legal_actions(board)}")

    new = board.copy()

    for r in range(ROWS - 1, -1, -1):
        if new[r, col] == EMPTY:
            new[r, col] = player
            return new

    raise ValueError(f"Columna llena: {col}")

def check_winner(board):
    directions = [
        (0, 1),
        (1, 0),
        (1, 1),
        (1, -1),
    ]

    for r in range(ROWS):
        for c in range(COLS):
            player = board[r, c]

            if player == EMPTY:
                continue

            for dr, dc in directions:
                ok = True

                for k in range(4):
                    rr = r + dr * k
                    cc = c + dc * k

                    if not (0 <= rr < ROWS and 0 <= cc < COLS and board[rr, cc] == player):
                        ok = False
                        break

                if ok:
                    return int(player)

    return 0

def is_terminal(board):
    return check_winner(board) != 0 or len(legal_actions(board)) == 0


## 4. Agentes auxiliares y wrapper configurable

`ConfiguredPolicy` permite cambiar variables de configuración sin modificar tus archivos originales.


In [ ]:
class RandomPolicy:
    def __init__(self, seed=0):
        self.seed = seed
        self.rng = np.random.default_rng(seed)

    def mount(self, timeout=None):
        self.rng = np.random.default_rng(self.seed)

    def act(self, board):
        return int(self.rng.choice(legal_actions(board)))


class CenterPolicy:
    def mount(self, timeout=None):
        pass

    def act(self, board):
        order = [3, 2, 4, 1, 5, 0, 6]
        legal = legal_actions(board)

        for c in order:
            if c in legal:
                return c

        return int(legal[0])


class ConfiguredPolicy:
    def __init__(self, policy_cls, name=None, **params):
        self.policy_cls = policy_cls
        self.name = name or policy_cls.__name__
        self.params = params
        self.inner = None

    def mount(self, timeout=None):
        self.inner = self.policy_cls()

        try:
            self.inner.mount(timeout)
        except TypeError:
            self.inner.mount()

        for key, value in self.params.items():
            setattr(self.inner, key, value)

    def act(self, board):
        if self.inner is None:
            self.mount()

        return int(self.inner.act(board))


def mount_policy(policy, timeout=None):
    if hasattr(policy, "mount"):
        try:
            policy.mount(timeout)
        except TypeError:
            policy.mount()


## 5. Motor de partidas

`evaluate_matchup` ejecuta varias partidas y alterna colores para evitar sesgo por empezar primero.


In [ ]:
@dataclass
class GameResult:
    winner: int
    reason: str
    moves: int
    p1_time: float
    p2_time: float
    p1_actions: int
    p2_actions: int


def play_game(first_policy, second_policy, timeout=None, seed=None, max_turns=42):
    if seed is not None:
        np.random.seed(seed)

    mount_policy(first_policy, timeout)
    mount_policy(second_policy, timeout)

    board = new_board()
    player = P1

    p1_time = 0.0
    p2_time = 0.0
    p1_actions = 0
    p2_actions = 0

    for turn in range(max_turns):
        policy = first_policy if player == P1 else second_policy
        legal = legal_actions(board)

        t0 = time.perf_counter()

        try:
            action = int(policy.act(board.copy()))
        except Exception as e:
            winner = P2 if player == P1 else P1
            return GameResult(winner, f"error: {e}", turn, p1_time, p2_time, p1_actions, p2_actions)

        elapsed = time.perf_counter() - t0

        if player == P1:
            p1_time += elapsed
            p1_actions += 1
        else:
            p2_time += elapsed
            p2_actions += 1

        if action not in legal:
            winner = P2 if player == P1 else P1
            return GameResult(winner, f"illegal action {action}", turn, p1_time, p2_time, p1_actions, p2_actions)

        board = apply_move(board, action, player)
        winner = check_winner(board)

        if winner != 0:
            return GameResult(winner, "normal", turn + 1, p1_time, p2_time, p1_actions, p2_actions)

        if len(legal_actions(board)) == 0:
            return GameResult(0, "draw", turn + 1, p1_time, p2_time, p1_actions, p2_actions)

        player *= -1

    return GameResult(0, "max_turns", max_turns, p1_time, p2_time, p1_actions, p2_actions)


def evaluate_matchup(name_a, policy_factory_a, name_b, policy_factory_b, n_games=30, timeout=None, alternate_colors=True):
    rows = []

    for i in range(n_games):
        if alternate_colors and i % 2 == 1:
            first_name, second_name = name_b, name_a
            first_policy = policy_factory_b(i)
            second_policy = policy_factory_a(i)
        else:
            first_name, second_name = name_a, name_b
            first_policy = policy_factory_a(i)
            second_policy = policy_factory_b(i)

        result = play_game(first_policy, second_policy, timeout=timeout, seed=i)

        if result.winner == P1:
            winner_name = first_name
        elif result.winner == P2:
            winner_name = second_name
        else:
            winner_name = "Draw"

        rows.append({
            "game": i + 1,
            "first_player": first_name,
            "second_player": second_name,
            "winner": winner_name,
            "winner_piece": result.winner,
            "reason": result.reason,
            "moves": result.moves,
            "first_avg_time": result.p1_time / max(result.p1_actions, 1),
            "second_avg_time": result.p2_time / max(result.p2_actions, 1),
        })

    return pd.DataFrame(rows)


def summarize_matchup(df, name_a, name_b):
    return pd.DataFrame([
        {
            "agent": name_a,
            "wins": int((df["winner"] == name_a).sum()),
            "draws": int((df["winner"] == "Draw").sum()),
            "losses": int((df["winner"] == name_b).sum()),
            "win_rate": float((df["winner"] == name_a).mean()),
            "avg_moves": float(df["moves"].mean()),
        },
        {
            "agent": name_b,
            "wins": int((df["winner"] == name_b).sum()),
            "draws": int((df["winner"] == "Draw").sum()),
            "losses": int((df["winner"] == name_a).sum()),
            "win_rate": float((df["winner"] == name_b).mean()),
            "avg_moves": float(df["moves"].mean()),
        },
    ])


def plot_win_rates(summary_df, title):
    plt.figure(figsize=(8, 5))
    plt.bar(summary_df["agent"], summary_df["win_rate"])
    plt.ylim(0, 1.05)
    plt.ylabel("Win rate")
    plt.title(title)
    plt.xticks(rotation=20)
    plt.grid(axis="y", alpha=0.3)
    plt.show()


## 6. Experimento 1 — V1 y V2 contra Random

Este experimento valida el desempeño contra el jugador aleatorio alternando colores.


In [ ]:
N_GAMES_RANDOM = 40

def make_v1(seed):
    return ConfiguredPolicy(RitaV1Class, name="Rita V1")

def make_v2(seed):
    return ConfiguredPolicy(RitaV2Class, name="Rita V2")

def make_random(seed):
    return RandomPolicy(seed=seed)

df_v1_random = evaluate_matchup(
    "Rita V1", make_v1,
    "Random", make_random,
    n_games=N_GAMES_RANDOM,
    timeout=1.0,
    alternate_colors=True,
)

df_v2_random = evaluate_matchup(
    "Rita V2", make_v2,
    "Random", make_random,
    n_games=N_GAMES_RANDOM,
    timeout=1.0,
    alternate_colors=True,
)

summary_random = pd.concat([
    summarize_matchup(df_v1_random, "Rita V1", "Random"),
    summarize_matchup(df_v2_random, "Rita V2", "Random"),
], ignore_index=True)

summary_random


In [ ]:
plot_win_rates(
    summary_random[summary_random["agent"].isin(["Rita V1", "Rita V2"])],
    "Win rate de Rita V1 y Rita V2 contra Random"
)


## 7. Experimento 2 — Desempeño por color


In [ ]:
def color_breakdown(df, agent_name):
    rows = []

    for role_col, role_name in [("first_player", "first/-1"), ("second_player", "second/1")]:
        played = df[df[role_col] == agent_name]

        if len(played) == 0:
            continue

        rows.append({
            "agent": agent_name,
            "role": role_name,
            "games": len(played),
            "wins": int((played["winner"] == agent_name).sum()),
            "draws": int((played["winner"] == "Draw").sum()),
            "losses": int(((played["winner"] != agent_name) & (played["winner"] != "Draw")).sum()),
            "win_rate": float((played["winner"] == agent_name).mean()),
            "avg_moves": float(played["moves"].mean()),
        })

    return pd.DataFrame(rows)

color_df = pd.concat([
    color_breakdown(df_v1_random, "Rita V1"),
    color_breakdown(df_v2_random, "Rita V2"),
], ignore_index=True)

color_df


In [ ]:
plt.figure(figsize=(8, 5))
labels = color_df["agent"] + " " + color_df["role"]
plt.bar(labels, color_df["win_rate"])
plt.ylim(0, 1.05)
plt.ylabel("Win rate")
plt.title("Desempeño por color / posición de turno")
plt.xticks(rotation=25)
plt.grid(axis="y", alpha=0.3)
plt.show()


## 8. Experimento 3 — Comparación directa V1 vs V2


In [ ]:
N_GAMES_V1_V2 = 40

df_v1_v2 = evaluate_matchup(
    "Rita V1", make_v1,
    "Rita V2", make_v2,
    n_games=N_GAMES_V1_V2,
    timeout=1.0,
    alternate_colors=True,
)

summary_v1_v2 = summarize_matchup(df_v1_v2, "Rita V1", "Rita V2")
summary_v1_v2


In [ ]:
plot_win_rates(summary_v1_v2, "Comparación directa: Rita V1 vs Rita V2")

plt.figure(figsize=(8, 5))
plt.hist(df_v1_v2["moves"], bins=range(0, 44, 2))
plt.xlabel("Número de movimientos")
plt.ylabel("Frecuencia")
plt.title("Distribución de duración de partidas: V1 vs V2")
plt.grid(axis="y", alpha=0.3)
plt.show()


## 9. Experimento 4 — Efecto de `exploration_c`

`exploration_c` controla el balance entre exploración y explotación en UCB.


In [ ]:
C_VALUES = [0.25, 0.5, 1.0, math.sqrt(2), 2.0, 3.0]
N_GAMES_C = 24

c_results = []

for c in C_VALUES:
    def make_v2_c(seed, c=c):
        return ConfiguredPolicy(
            RitaV2Class,
            name=f"Rita V2 c={c:.2f}",
            exploration_c=c,
            num_iterations=800,
            time_limit=0.75,
        )

    agent_name = f"Rita V2 c={c:.2f}"

    df_c = evaluate_matchup(
        agent_name,
        make_v2_c,
        "Random",
        make_random,
        n_games=N_GAMES_C,
        timeout=1.0,
        alternate_colors=True,
    )

    c_results.append({
        "c": c,
        "win_rate": float((df_c["winner"] == agent_name).mean()),
        "draw_rate": float((df_c["winner"] == "Draw").mean()),
        "loss_rate": float(((df_c["winner"] != agent_name) & (df_c["winner"] != "Draw")).mean()),
        "avg_moves": float(df_c["moves"].mean()),
    })

df_c_results = pd.DataFrame(c_results)
df_c_results


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(df_c_results["c"], df_c_results["win_rate"], marker="o")
plt.ylim(0, 1.05)
plt.xlabel("exploration_c")
plt.ylabel("Win rate contra Random")
plt.title("Efecto de exploration_c en Rita V2")
plt.grid(alpha=0.3)
plt.show()


## 10. Experimento 5 — Cantidad de rollouts / iteraciones


In [ ]:
ITERATION_VALUES = [100, 250, 500, 800, 1200]
N_GAMES_ITERS = 20

iteration_results = []

for n_iter in ITERATION_VALUES:
    def make_v2_iter(seed, n_iter=n_iter):
        return ConfiguredPolicy(
            RitaV2Class,
            name=f"Rita V2 iter={n_iter}",
            num_iterations=n_iter,
            time_limit=0.85,
            exploration_c=math.sqrt(2),
        )

    agent_name = f"Rita V2 iter={n_iter}"

    df_iter = evaluate_matchup(
        agent_name,
        make_v2_iter,
        "Random",
        make_random,
        n_games=N_GAMES_ITERS,
        timeout=1.0,
        alternate_colors=True,
    )

    time_values = []
    time_values.extend(df_iter.loc[df_iter["first_player"] == agent_name, "first_avg_time"].tolist())
    time_values.extend(df_iter.loc[df_iter["second_player"] == agent_name, "second_avg_time"].tolist())

    iteration_results.append({
        "num_iterations": n_iter,
        "win_rate": float((df_iter["winner"] == agent_name).mean()),
        "loss_rate": float(((df_iter["winner"] != agent_name) & (df_iter["winner"] != "Draw")).mean()),
        "avg_moves": float(df_iter["moves"].mean()),
        "avg_decision_time": float(np.mean(time_values)) if time_values else np.nan,
    })

df_iter_results = pd.DataFrame(iteration_results)
df_iter_results


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(df_iter_results["num_iterations"], df_iter_results["win_rate"], marker="o")
plt.ylim(0, 1.05)
plt.xlabel("num_iterations")
plt.ylabel("Win rate contra Random")
plt.title("Efecto del número de rollouts en Rita V2")
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(df_iter_results["num_iterations"], df_iter_results["avg_decision_time"], marker="o")
plt.xlabel("num_iterations")
plt.ylabel("Tiempo promedio por decisión (s)")
plt.title("Costo temporal al aumentar rollouts")
plt.grid(alpha=0.3)
plt.show()


## 11. Ablation study — aporte de subestrategias

Se compara V2 completa contra versiones donde se desactiva una mejora.


In [ ]:
class V2NoSafeActions(ConfiguredPolicy):
    def mount(self, timeout=None):
        super().mount(timeout)

        def no_filter(board, player, actions):
            return actions.copy()

        self.inner.get_safe_actions = no_filter


class V2NoDoubleThreat(ConfiguredPolicy):
    def mount(self, timeout=None):
        super().mount(timeout)

        def no_double(board, player, candidate_actions):
            return None

        self.inner.find_double_threat_move = no_double


class V2LessRollouts(ConfiguredPolicy):
    def mount(self, timeout=None):
        super().mount(timeout)
        self.inner.num_iterations = 350
        self.inner.time_limit = 0.7


ABLATIONS = [
    ("V2 completa", lambda seed: ConfiguredPolicy(RitaV2Class, name="V2 completa")),
    ("V2 sin filtro seguro", lambda seed: V2NoSafeActions(RitaV2Class, name="V2 sin filtro seguro")),
    ("V2 sin amenazas dobles", lambda seed: V2NoDoubleThreat(RitaV2Class, name="V2 sin amenazas dobles")),
    ("V2 con rollouts V1", lambda seed: V2LessRollouts(RitaV2Class, name="V2 con rollouts V1")),
]

N_GAMES_ABLATION = 20
ablation_results = []

for name, factory in ABLATIONS:
    df_ab = evaluate_matchup(
        name,
        factory,
        "Random",
        make_random,
        n_games=N_GAMES_ABLATION,
        timeout=1.0,
        alternate_colors=True,
    )

    ablation_results.append({
        "version": name,
        "win_rate": float((df_ab["winner"] == name).mean()),
        "draw_rate": float((df_ab["winner"] == "Draw").mean()),
        "loss_rate": float(((df_ab["winner"] != name) & (df_ab["winner"] != "Draw")).mean()),
        "avg_moves": float(df_ab["moves"].mean()),
    })

df_ablation = pd.DataFrame(ablation_results)
df_ablation


In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(df_ablation["version"], df_ablation["win_rate"])
plt.ylim(0, 1.05)
plt.ylabel("Win rate contra Random")
plt.title("Ablation study: aporte de subestrategias en Rita V2")
plt.xticks(rotation=25, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.show()


## 12. Conclusiones automáticas


In [ ]:
print("=== Conclusiones preliminares ===")

try:
    v1_wr = float(summary_random.loc[summary_random["agent"] == "Rita V1", "win_rate"].iloc[0])
    v2_wr = float(summary_random.loc[summary_random["agent"] == "Rita V2", "win_rate"].iloc[0])

    print(f"Win rate V1 vs Random: {v1_wr:.2%}")
    print(f"Win rate V2 vs Random: {v2_wr:.2%}")

    if v2_wr > v1_wr:
        print("- V2 mejora el desempeño contra Random.")
    elif v2_wr == v1_wr:
        print("- V1 y V2 tienen el mismo win rate contra Random; revisar V1 vs V2, tiempo y estabilidad.")
    else:
        print("- V2 no superó a V1 contra Random; revisar calibración de heurísticas.")
except Exception as e:
    print("No se pudo leer summary_random:", e)

try:
    best_c_row = df_c_results.sort_values("win_rate", ascending=False).iloc[0]
    print(f"- Mejor c observado: {best_c_row['c']:.3f} con win rate {best_c_row['win_rate']:.2%}.")
except Exception as e:
    print("No se pudo leer df_c_results:", e)

try:
    best_iter_row = df_iter_results.sort_values(["win_rate", "avg_decision_time"], ascending=[False, True]).iloc[0]
    print(f"- Mejor configuración de iteraciones observada: {int(best_iter_row['num_iterations'])} rollouts.")
    print(f"  Win rate: {best_iter_row['win_rate']:.2%}, tiempo promedio: {best_iter_row['avg_decision_time']:.4f}s.")
except Exception as e:
    print("No se pudo leer df_iter_results:", e)


## 13. Texto base para PDF o presentación

> La versión V1 implementa MCTS con selección UCB y rollouts aleatorios. Esta versión usa reglas tácticas previas para ganar o bloquear victorias inmediatas, pero su principal cuello de botella es que la default policy en la simulación no tiene conocimiento del juego.  
> La versión V2 conserva MCTS + UCB, pero mejora la calidad de las simulaciones: aumenta el número de rollouts, ordena la expansión por heurística, evita jugadas suicidas, detecta amenazas dobles y evalúa tableros no terminales usando ventanas de cuatro posiciones.  
> Experimentalmente, se evaluó el desempeño contra un jugador aleatorio, el efecto del color, la comparación directa V1 vs V2, el impacto de `exploration_c` y el número de rollouts.
